# General phase-matching pipeline

Repeatable version of the ad-hoc `PhaseMatching_Table.ipynb` investigation, built around one
lesson learned the hard way there: **raw mode index is not a stable identifier across geometry or
wavelength** (a mode's index has been observed to swing by 7+ just from a 50nm width change, and a
single mislabeled mode -- TM13 mistaken for TM40 -- cost an entire session of debugging). So the
workflow here puts ALL human visual confirmation up front, at one anchor geometry, and everything
after that runs unattended (potentially for hours) using overlap-based tracking that walks between
neighboring grid points rather than jumping straight from the anchor to every target.

**Workflow:**
1. Set the anchor geometry and materials.
2. Visually survey modes at the anchor (`visual_mode_survey`) -- review the plots, then fill in
   `modes_of_interest` with the confirmed index for each named mode.
3. (Optional) Auto-scan for *other* modes' crossings at the anchor (`scan_for_crossings`) -- surfaces
   candidates like TM02/TM20 without needing to know their index ahead of time. Visually confirm any
   candidates you want to add before including them in step 4.
4. Define the target geometry grid to cover.
5. Long automated run: for each named mode, walk its mode index across the whole grid
   (`walk_mode_across_points`), then compute the phase-matching crossing wavelength and
   width/height gradients at every grid point (`phase_matching_row`).
6. Assemble results into a table and save to CSV.

In [1]:
import numpy as np
import itertools
import emode_helpers as eh

# --- Anchor geometry + materials (edit for your design) ---
h_anchor = 350.0  # [nm]
w_anchor = 400.0  # [nm]
dx, dy = 10.0, 10.0

eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"

## Step 1: visual mode survey at the anchor

Solves once at the anchor and plots each mode in `modes_to_plot` for you to review. Adjust the
range/wavelength below to cover whatever modes you're interested in -- widen it if the mode you
want isn't in the initial range. This is the ONLY step that needs your eyes on a plot; everything
after this cell is unattended.

In [2]:
survey_wavelength = 240.0  # [nm] -- pick something in the general region of interest
modes_to_plot = range(18, 35)  # narrowed for smoke test -- widen to range(0, 40) for a real new geometry

fdm_survey = eh.visual_mode_survey(anisotropic_equation, h_anchor, w_anchor, survey_wavelength,
                                    num_modes=45, modes_to_plot=modes_to_plot,
                                    x_resolution=dx, y_resolution=dy)

-- mode 18 --  n_eff=2.13200
-- mode 19 --  n_eff=2.12873
-- mode 20 --  n_eff=2.07072
-- mode 21 --  n_eff=2.06407
-- mode 22 --  n_eff=2.05721
-- mode 23 --  n_eff=2.04847
-- mode 24 --  n_eff=2.02615
-- mode 25 --  n_eff=2.00248
-- mode 26 --  n_eff=1.99444
-- mode 27 --  n_eff=1.97830
-- mode 28 --  n_eff=1.96695
-- mode 29 --  n_eff=1.95420
-- mode 30 --  n_eff=1.91851
-- mode 31 --  n_eff=1.88705
-- mode 32 --  n_eff=1.87591
-- mode 33 --  n_eff=1.86461
-- mode 34 --  n_eff=1.86113


## Step 1b: record confirmed mode indices

After reviewing the plots above, fill in the confirmed index for each named mode of interest.
`wavelengths_shg` should be wide enough to contain that mode's crossing across the ENTIRE target
grid you plan to sweep in step 4 -- not just at the anchor. A mode with a large `dWL/dwidth` (like
TM40, ~0.24 nm/nm) can shift its crossing wavelength by tens of nm across a wide grid, so give it a
generous window; a less width-sensitive mode (like TM04, ~-0.012 nm/nm) can use a narrower one.

`num_modes`/`window` for the walk should have headroom above the anchor index, since index has
been observed to shift by 7+ over a single 50nm step -- err generous, the walk is cheap per step.

In [4]:
modes_of_interest = {
    'TM04': dict(anchor_mode_idx=30, num_modes=45, window=15,
                 wavelengths_shg=np.arange(210, 251, 2.0), tracking_wavelength=228.244),
    'TM40': dict(anchor_mode_idx=23, num_modes=45, window=15,
                 wavelengths_shg=np.arange(200, 300, 2.0), tracking_wavelength=241.672),
}

## Step 2 (optional): auto-scan for other modes' crossings at the anchor

Looks for a phase-matching crossing across EVERY raw mode index at the anchor point, in exactly 2
EMode sweep calls (not one per index). This surfaces *candidates* like TM02/TM20 -- it does NOT
identify them by name. Visually confirm any index you want to keep (via `visual_mode_survey` with
`modes_to_plot=[that index]`) before adding it to `modes_of_interest` above and re-running from
Step 1b.

In [ ]:
scan_wavelengths = np.arange(160, 340, 2.0)
already_named = {cfg['anchor_mode_idx'] for cfg in modes_of_interest.values()}

candidates = eh.scan_for_crossings(anisotropic_equation, h_anchor, w_anchor, scan_wavelengths,
                                    fund_mode_idx=0, num_modes_fund=2, num_modes_scan=45,
                                    exclude_indices=already_named | {0},
                                    x_resolution=dx, y_resolution=dy)

print(f"Found {len(candidates)} candidate mode(s) with a crossing in "
      f"{scan_wavelengths[0]:.0f}-{scan_wavelengths[-1]:.0f}nm (excluding already-named modes):")
for mode_idx, info in sorted(candidates.items()):
    print(f"  mode {mode_idx}: crossing={info['wavelength_shg']:.3f}nm, n_eff={info['n_eff']:.5f}")

## Step 3: define the target geometry grid

Edit `h_values`/`w_values` to whatever range you want covered. The walk in Step 4 handles an
arbitrary set of (h, w) points, not just a rectangular grid, if you'd rather build `target_points`
some other way.

In [5]:
# Smoke-test grid -- just 2 extra points beyond the anchor, to validate the pipeline cheaply
# before committing to a big sweep. Widen back out once this runs clean end-to-end.
h_values = [345.0, 350.0, 355.0]
w_values = [400.0]

target_points = [(h, w) for h, w in itertools.product(h_values, w_values)
                  if (h, w) != (h_anchor, w_anchor)]
print(f"{len(target_points)} target points queued.")

2 target points queued.


## Step 4: long run -- walk mode index across the grid, for each named mode

This is the unattended, potentially-hours-long part. Resilient to individual point failures (a
failed point doesn't stop the rest); progress prints as it goes so you can check in without
watching continuously.

In [ ]:
walked_indices = {}  # mode_name -> {point: {'mode_idx', 'overlap', ...} or None}

for mode_name, cfg in modes_of_interest.items():
    print(f"\n=== Walking {mode_name} across {len(target_points)} points ===")
    walked_indices[mode_name] = eh.walk_mode_across_points(
        anisotropic_equation, (h_anchor, w_anchor), cfg['anchor_mode_idx'], target_points,
        tracking_wavelength=cfg['tracking_wavelength'], num_modes=cfg['num_modes'],
        window=cfg['window'], x_resolution=dx, y_resolution=dy)


=== Walking TM04 across 2 points ===


## Step 5: phase-matching crossing + width/height gradient at every point

Uses each point's tracked mode index (from Step 4) directly -- no further tracking uncertainty.
Points where the walk failed or landed on a saturated (num_modes-ceiling) index are skipped with a
printed note rather than silently dropped.

In [ ]:
all_rows = []
total_considered = 0

for mode_name, cfg in modes_of_interest.items():
    # the anchor point itself, using the originally-confirmed index
    points_and_indices = [((h_anchor, w_anchor), cfg['anchor_mode_idx'])]
    points_and_indices += [(p, info['mode_idx']) for p, info in walked_indices[mode_name].items()
                           if info is not None and not info['saturated']]
    total_considered += len(points_and_indices)

    for (h, w), mode_idx in points_and_indices:
        print(f"{mode_name} @ h={h}, w={w}, mode={mode_idx} ...")
        try:
            row = eh.phase_matching_row(
                anisotropic_equation, h, w, cfg['wavelengths_shg'], target_mode_idx=mode_idx,
                target_mode_name=mode_name, fund_mode_idx=0, num_modes_fund=6,
                num_modes_target=mode_idx + 10, x_resolution=dx, y_resolution=dy)
            if row is None:
                print(f"  no crossing found in {cfg['wavelengths_shg'][0]:.0f}-"
                      f"{cfg['wavelengths_shg'][-1]:.0f}nm -- skipped")
            else:
                all_rows.append(row)
        except Exception as e:
            print(f"  FAILED: {repr(e)}")

print(f"\n{len(all_rows)} of {total_considered} points computed successfully.")

## Step 6: results table + save

In [ ]:
print(eh.format_table(all_rows))

import csv
csv_path = 'general_pipeline_results.csv'
if all_rows:
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"\nSaved {len(all_rows)} rows to {csv_path}")